# SFT LoRA with Cut Tokenizer (Qwen3-4B-Thinking)

This notebook mirrors NVARC's tokenizer cutting approach, then runs LoRA fine-tuning and evaluation on the cut model.

## Configuration

In [1]:
import os
print(os.getcwd())
os.chdir("/home/ARC")

/home/ARC/ARChitects


In [2]:
# Config
from dataclasses import dataclass
import os
import random
import torch

@dataclass
class Config:
    model_path: str = 'models/Qwen3-4B-Thinking-2507'
    dataset_path: str = 'SDG/data/arc2_evaluation'  # HF dataset with messages, for tokenizer cutting
    chat_template_path: str = 'SDG/chat_template.j2'
    cut_output_dir: str = 'models/Qwen3-4B-Thinking-2507'
    max_vocab_size: int = 16

    sdg_dataset_paths: tuple = (
        'SDG/data/nvarc_training',
        'SDG/data/arc2_training',
        'SDG/data/rearc',
    )
    use_streaming: bool = False
    shuffle_buffer: int = 10000
    sample_size: int = 0
    sample_ratio: float = 0.0
    sample_seed: int = 42

    eval_root: str = 'data'
    eval_split: str = 'evaluation'
    max_eval_tasks: int = 10

    max_seq_len: int = 1024
    max_new_tokens: int = 512

    lora_r: int = 64
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = (
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    )

    output_dir: str = 'outputs/qwen3-4b-thinking-16tok-cut-multi-stream'
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-4
    num_train_epochs: int = 3
    logging_steps: int = 10
    save_steps: int = 200

cfg = Config()

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.sample_seed)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

device: cuda


## Preparation

In [ ]:
!kaggle datasets download -d sorokin/nvarc-synthetic-puzzles
!unzip -o nvarc-synthetic-puzzles.zip -d SDG/NVARCsynthetic

In [3]:
# Cut tokenizer vocab based on message dataset (NVARC-style)
from transformers import AutoTokenizer
from datasets import load_from_disk
import json

tokenizer = AutoTokenizer.from_pretrained(cfg.model_path)
with open(cfg.chat_template_path, 'r', encoding='utf-8') as f:
    tokenizer.chat_template = f.read()

ds = load_from_disk(cfg.dataset_path)

def get_tokens(sample):
    text = tokenizer.apply_chat_template(sample['messages'], tokenize=False)
    tokens = tokenizer.encode(text + '<|endoftext|>')
    sample['tokens'] = tokens
    return sample

ds = ds.map(get_tokens)

all_tokens = []
for sample in ds:
    all_tokens.extend(sample['tokens'])
all_tokens = sorted(set(all_tokens))

token_strs = [tokenizer.convert_ids_to_tokens(t) for t in all_tokens]
print('token_strs:', token_strs)

expected = set(['<|im_start|>', '<|im_end|>', '<|endoftext|>', '\n', 'user', 'assistant'] + [str(i) for i in range(10)])
if set(token_strs) != expected:
    print('WARNING: token set differs from expected')
    print('expected:', expected)
    print('actual:', set(token_strs))

if cfg.max_vocab_size and len(all_tokens) > cfg.max_vocab_size:
    raise ValueError(f'token count {len(all_tokens)} exceeds max_vocab_size {cfg.max_vocab_size}')

mapping = []
new_vocab = {}
for token_id in all_tokens:
    tok = tokenizer.convert_ids_to_tokens(token_id)
    new_vocab[tok] = len(mapping)
    mapping.append(token_id)

print('vocab size:', len(new_vocab))

os.makedirs(cfg.cut_output_dir, exist_ok=True)
with open(os.path.join(cfg.cut_output_dir, 'mapping.json'), 'w', encoding='utf-8') as f:
    json.dump({'old_ids': mapping, 'tokens': list(new_vocab.keys())}, f, ensure_ascii=False, indent=2)

token_strs: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'Ċ', 'user', 'assistant', '<|endoftext|>', '<|im_start|>', '<|im_end|>']
expected: {'<|im_end|>', 'assistant', 'user', '<|im_start|>', '8', '<|endoftext|>', '9', '0', '\n', '1', '5', '3', '4', '6', '2', '7'}
actual: {'assistant', '<|im_end|>', '8', 'user', '<|im_start|>', 'Ċ', '9', '<|endoftext|>', '0', '1', '5', '3', '4', '6', '2', '7'}
vocab size: 16


In [4]:
# Build SFT training samples from multiple SDG augmented datasets
from datasets import load_from_disk, concatenate_datasets, interleave_datasets
from torch.utils.data import IterableDataset

all_datasets = []
for path in cfg.sdg_dataset_paths:
    ds = load_from_disk(path)
    all_datasets.append(ds)
    print(path, ds)

if not all_datasets:
    raise ValueError('no datasets loaded')

if cfg.use_streaming:
    stream_sets = []
    for ds in all_datasets:
        stream_sets.append(ds.to_iterable_dataset())
    ds = interleave_datasets(stream_sets)
    if cfg.shuffle_buffer and cfg.shuffle_buffer > 0:
        ds = ds.shuffle(buffer_size=cfg.shuffle_buffer, seed=cfg.sample_seed)
    if cfg.sample_size and cfg.sample_size > 0:
        ds = ds.take(cfg.sample_size)
    elif cfg.sample_ratio and cfg.sample_ratio > 0:
        raise ValueError('sample_ratio is not supported with streaming; use sample_size')
else:
    if len(all_datasets) == 1:
        ds = all_datasets[0]
    else:
        ds = concatenate_datasets(all_datasets)
    if cfg.sample_ratio and cfg.sample_ratio > 0:
        n = int(len(ds) * cfg.sample_ratio)
        if n < 1:
            raise ValueError('sample_ratio too small for dataset size')
        ds = ds.shuffle(seed=cfg.sample_seed).select(range(n))
    elif cfg.sample_size and cfg.sample_size > 0:
        ds = ds.shuffle(seed=cfg.sample_seed).select(range(cfg.sample_size))

print('streaming:', cfg.use_streaming)
if not cfg.use_streaming:
    print('total training samples:', len(ds))

def iter_samples(dataset):
    for row in dataset:
        msgs = row["messages"]
        last_idx = max(i for i, m in enumerate(msgs) if m["role"] == "assistant")
        prompt_msgs = msgs[:last_idx]
        target = msgs[last_idx]["content"]

        parts = []
        for m in prompt_msgs:
            parts.append("<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        prompt = "".join(parts)
        yield {"prompt": prompt, "target": target}

class StreamSFTDataset(IterableDataset):
    def __init__(self, dataset, tokenizer, max_seq_len: int):
        self._dataset = dataset
        self._tokenizer = tokenizer
        self._max_seq_len = max_seq_len

    def __iter__(self):
        for item in iter_samples(self._dataset):
            prompt_ids = self._tokenizer(item["prompt"], add_special_tokens=False)["input_ids"]
            target_ids = self._tokenizer(item["target"], add_special_tokens=False)["input_ids"]

            eos = self._tokenizer.eos_token_id
            input_ids = prompt_ids + target_ids + ([eos] if eos is not None else [])
            labels = [-100] * len(prompt_ids) + target_ids + ([eos] if eos is not None else [])

            if len(input_ids) > self._max_seq_len:
                input_ids = input_ids[-self._max_seq_len :]
                labels = labels[-self._max_seq_len :]

            yield {
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            }

SDG/data/nvarc_training Dataset({
    features: ['puzzle_name', 'messages'],
    num_rows: 1131869
})
SDG/data/arc2_training Dataset({
    features: ['puzzle_name', 'messages'],
    num_rows: 155904
})
SDG/data/rearc Dataset({
    features: ['puzzle_name', 'messages'],
    num_rows: 102392
})
streaming: False
total training samples: 1390165


In [5]:
# Load cut model and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(cfg.cut_output_dir, use_fast=False)
with open(cfg.chat_template_path, 'r', encoding='utf-8') as f:
    tokenizer.chat_template = f.read()

if tokenizer.eos_token_id is None:
    eos_id = tokenizer.convert_tokens_to_ids('<|endoftext|>')
    if eos_id is not None and eos_id != tokenizer.unk_token_id:
        tokenizer.eos_token_id = eos_id

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else '<|endoftext|>'

model = AutoModelForCausalLM.from_pretrained(
    cfg.cut_output_dir,
    dtype=getattr(torch, 'bfloat16', None),
    device_map='auto',
)
model.config.use_cache = False

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Pre-evalution

In [7]:
# Compute max_new_tokens based on 30x30 grid
from eval.core import GridCodec

codec = GridCodec()
max_grid = [[0 for _ in range(30)] for _ in range(30)]
reply_text = codec.grid_to_text(max_grid) + '<|im_end|>'
max_new_tokens = len(tokenizer.encode(reply_text)) + 1
print('max_new_tokens:', max_new_tokens)

max_new_tokens: 931


In [8]:
# Baseline evaluation with cut model (before LoRA)
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core import GridCodec, ARCDataset, run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

codec = GridCodec()
eval_dataset = ARCDataset(root=cfg.eval_root, split=cfg.eval_split, max_tasks=cfg.max_eval_tasks)
# Preview a couple of eval tasks
for i, task in enumerate(eval_dataset):
    if i >= 10:
        break
    print('--- task', task.get('task_id', i))
    print('train len:', len(task.get('train', [])), 'test len:', len(task.get('test', [])))
    print('sample train input:', task.get('train', [{}])[0].get('input'))
    print('sample train output:', task.get('train', [{}])[0].get('output'))


--- task 0934a4d8
train len: 4 test len: 1
sample train input: [[3, 5, 3, 3, 6, 6, 5, 4, 1, 4, 9, 9, 4, 3, 9, 9, 9, 9, 3, 4, 9, 9, 4, 1, 4, 5, 6, 6, 3, 3], [5, 3, 3, 3, 6, 6, 4, 5, 4, 1, 9, 9, 3, 4, 9, 1, 1, 9, 4, 3, 9, 9, 1, 4, 5, 4, 6, 6, 3, 3], [1, 1, 3, 5, 5, 4, 6, 6, 9, 1, 1, 4, 9, 9, 4, 5, 5, 4, 9, 9, 4, 1, 1, 9, 6, 6, 4, 5, 5, 3], [1, 1, 5, 3, 4, 5, 6, 6, 1, 9, 4, 1, 9, 1, 4, 4, 4, 4, 1, 9, 1, 4, 9, 1, 6, 6, 5, 4, 3, 5], [6, 9, 9, 9, 3, 5, 3, 3, 4, 3, 9, 9, 9, 2, 6, 9, 9, 6, 2, 9, 9, 9, 3, 4, 3, 3, 5, 3, 9, 9], [9, 6, 9, 9, 5, 3, 3, 3, 3, 4, 9, 1, 9, 9, 9, 6, 6, 9, 9, 9, 1, 9, 4, 3, 3, 3, 3, 5, 9, 9], [9, 9, 6, 9, 1, 1, 3, 5, 9, 9, 4, 4, 6, 9, 9, 2, 2, 9, 9, 6, 4, 4, 9, 9, 5, 3, 1, 1, 9, 6], [9, 9, 9, 6, 1, 1, 5, 3, 9, 1, 5, 4, 9, 6, 9, 9, 9, 9, 6, 9, 4, 5, 1, 9, 3, 5, 1, 1, 6, 9], [1, 4, 9, 1, 4, 3, 9, 9, 5, 5, 7, 2, 4, 3, 2, 4, 4, 2, 3, 4, 2, 7, 5, 5, 9, 9, 3, 4, 1, 9], [4, 1, 1, 9, 3, 4, 9, 1, 4, 5, 2, 7, 3, 4, 4, 2, 2, 4, 4, 3, 7, 2, 5, 4, 1, 9, 4, 3, 9, 1], [9, 9, 1, 4, 9, 

In [9]:
baseline_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
baseline_solver = RawSolver(model=baseline_model, codec=codec)
baseline_reports = run_evaluation(
    dataset=eval_dataset,
    solver=baseline_solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports_baseline'),
    model_id=cfg.cut_output_dir,
    model_key='baseline_cut',
    viz_failures=True,
    max_tasks=cfg.max_eval_tasks,
)
print(baseline_reports['summary'])

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[eval] task=0934a4d8 status=wrong
[eval] task=135a2760 status=wrong
[eval] task=136b0064 status=wrong
[eval] task=13e47133 status=wrong
[eval] task=142ca369 status=wrong
[eval] task=16b78196 status=wrong
[eval] task=16de56c4 status=wrong
[eval] task=1818057f status=wrong
[eval] task=195c6913 status=wrong
[eval] task=1ae2feb7 status=wrong
{'accuracy': 0.0, 'scored': 10, 'total': 10}


## LoRA

In [15]:
print(torch.cuda.memory_summary())
print(torch.cuda.memory_allocated() / 1024**2, "MB")

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 1         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  44236 MiB |  44673 MiB | 397387 MiB | 353151 MiB |
|       from large pool |  43623 MiB |  44097 MiB | 395269 MiB | 351645 MiB |
|       from small pool |    612 MiB |    614 MiB |   2117 MiB |   1505 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  44236 MiB |  44673 MiB | 397387 MiB | 353151 MiB |
|       from large pool |  43623 MiB |  44097 MiB | 395269 MiB |

In [6]:
# Build PyTorch Dataset and collator
from ARChitects.sft_utils import ArcSFTDataset, collate_sft

if cfg.use_streaming:
    train_dataset = StreamSFTDataset(ds, tokenizer, max_seq_len=cfg.max_seq_len)
else:
    samples = list(iter_samples(ds))
    train_dataset = ArcSFTDataset(samples, tokenizer, max_seq_len=cfg.max_seq_len)

def collate_fn(batch):
    return collate_sft(batch, tokenizer=tokenizer)

In [7]:
# Freeze base; train LoRA only
from peft import LoraConfig, get_peft_model

for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules),
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


In [ ]:
# Train LoRA
from transformers import TrainingArguments, Trainer

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    bf16=bf16_ok,
    fp16=torch.cuda.is_available() and not bf16_ok,
    report_to='none',
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

trainer.train()

Step,Training Loss
10,0.290769
20,0.192597
30,0.155276
40,0.142672
50,0.199910
60,0.208994
70,0.201268
80,0.219809
90,0.228004
100,0.255279


In [ ]:
# Save LoRA adapter
adapter_dir = os.path.join(cfg.output_dir, 'lora_adapter')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('saved:', adapter_dir)

## Evalution

In [ ]:
# Evaluate with eval/ and write reports
eval_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
solver = RawSolver(model=eval_model, codec=codec)

reports = run_evaluation(
    dataset=eval_dataset,
    solver=solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports'),
    model_id=cfg.cut_output_dir,
    model_key='lora_sft_cut',
    viz_failures=True,
)

print(reports['summary'])

In [ ]:
# Save Lora-ed model as a merged one
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(cfg.cut_output_dir, device_map='auto')
lora = PeftModel.from_pretrained(base, os.path.join(cfg.output_dir, 'lora_adapter'))
merged = lora.merge_and_unload()
merged.save_pretrained(os.path.join(cfg.output_dir, 'merged'))
tokenizer.save_pretrained(os.path.join(cfg.output_dir, 'merged'))